引入 **Z-Score 标准化**（也叫分数标准化）是量化从“初级脚本”走向“工业级策略”的必经之路。

### 1. 为什么要引入 Z-Score？（解决“死板”问题）

你之前的阈值是“死”的（比如 $X_1 < -1\%$）。
*   **问题：** 对于**平安银行**，跌 1% 可能是发生了特大利空，橡皮筋可能断了（不会回弹）；对于**通达股份**，跌 1% 可能只是正常的日常抖动，还没拉到反弹的程度。
*   **核心逻辑：** 我们不应该看它跌了百分之几，而应该看它**“跌掉了几个标准差”**。

### 2. Z-Score 的数学定义

对于某个特征 $X$（比如 $X_1$），它在 $t$ 时刻的 Z-Score 计算如下：

$$Z_t = \frac{X_t - \mu}{\sigma}$$

其中：
*   **$X_t$**：当前的特征值（比如今天 10:30 的 Price/VWAP-1）。
*   **$\mu$ (均值)**：该股票过去 $N$ 天在 10:30 这个时刻该特征的平均值（通常接近 0）。
*   **$\sigma$ (标准差)**：该股票过去 $N$ 天在 10:30 这个时刻该特征的波动幅度。

---

### 3. 具体算法实现步骤

假设你设定回顾窗口 $N=20$（即参考过去一个月的历史）：

#### 第一步：建立历史基准线
你需要计算每只股票过去 20 个交易日，在 **10:30** 这个时间点的 $X_1$ 和 $X_2$。
*   你得到两个序列：`History_X1 = [x1_day1, x1_day2, ..., x1_day20]`
*   计算均值 $\mu_{X1}$ 和标准差 $\sigma_{X1}$。

#### 第二步：计算实时信号（归一化）
今天 10:30，你算出了今天的 $X_{1\_today}$：
*   $Z_{X1} = \frac{X_{1\_today} - \mu_{X1}}{\sigma_{X1}}$
*   $Z_{X2} = \frac{X_{2\_today} - \mu_{X2}}{\sigma_{X2}}$

#### 第三步：构建归一化后的组合因子 $Z_{final}$
由于 $Z_{X1}$ 和 $Z_{X2}$ 已经是无量纲的数字了（它们都在 0 附近，绝大多数在 -3 到 3 之间），你可以放心地合并它们：
$$Z_{final} = Z_{X1} + w \cdot Z_{X2}$$
（比如继续沿用你的经验，给 $w=2$）

---

### 4. Z-Score 后的“共振策略”怎么写？

引入 Z-Score 后，你的策略会变得非常优雅：

*   **超跌共振触发：** 当 $Z_{X1} < -2.0$ 且 $Z_{X2} < -2.0$。
    *   **含义：** “今天该股票的价格偏离度和能量偏离度，**同时**达到了其自身历史波动的 2 倍标准差以外。”
    *   **统计学意义：** 这属于该股票自身 95% 概率之外的极端异常情况，均值回归的动能极强。

---

### 5. 程序员的 Python 实现逻辑

你可以利用 `pandas` 的滚动窗口函数快速实现：

```python
import pandas as pd

# 假设 df 是你所有股票 10:30 时刻的历史记录
# window = 20 天

# 1. 计算 X1 和 X2 (你已经写好了)
df['X1'] = df['price'] / df['vwap'] - 1
df['X2'] = df['vwap'] / df['twap'] - 1

# 2. 计算滚动均值和标准差 (核心代码)
# 注意：一定要用 shift(1)，避免用到当天的信息导致回测死鱼
df['X1_mu'] = df.groupby('ticker')['X1'].transform(lambda x: x.rolling(20).mean().shift(1))
df['X1_std'] = df.groupby('ticker')['X1'].transform(lambda x: x.rolling(20).std().shift(1))

df['X2_mu'] = df.groupby('ticker')['X2'].transform(lambda x: x.rolling(20).mean().shift(1))
df['X2_std'] = df.groupby('ticker')['X2'].transform(lambda x: x.rolling(20).std().shift(1))

# 3. 计算 Z-Score
df['Z_X1'] = (df['X1'] - df['X1_mu']) / df['X1_std']
df['Z_X2'] = (df['X2'] - df['X2_mu']) / df['X2_std']

# 4. 组合因子
df['Z_final'] = df['Z_X1'] + 2 * df['Z_X2']
```

---

### 6. 为什么这样做会增强你的效果？

1.  **自动适配波动率：** 平安银行跌 0.3% 就能算出 Z = -2.5；通达股份要跌 1.5% 才能算出 Z = -2.5。你的模型会自动在不同票上寻找**相对极端**的点。
2.  **消除量纲影响：** $X_1$ 的量级通常比 $X_2$ 大。直接 $X_1 + 2 \cdot X_2$ 可能导致 $X_1$ 占主导。标准化后，两者的“力量”是在同一个维度比较，加权才真正有意义。
3.  **横向可比性：** 100 只持仓股，你可以算出 100 个 $Z_{final}$。每天只挑 $Z_{final}$ 最低（最超跌）的那 5 只票做T。这种**“截面比惨”**的逻辑，胜率比死守单只票高得多。

### 下一步建议：
你可以尝试把 002591 的 $X_1, X_2$ 转化成 $Z_{X1}, Z_{X2}$ 后，再看 $Z_{final}$ 与 $Y$ 的相关性。我打赌相关系数会进一步提升，因为你剔除了个股自身波动水平的噪音。

你想看看如何根据这个 Z-Score 建立一个**“每日做T候选名单筛选器”**吗？